# ParkCatalog Demo

Two methods: `search()` to find parks, `what_is_available()` to see what Cal-Adapt data exists.

```bash
./docker/run_docker.sh build     # first time
./docker/run_docker.sh notebook  # then this
```

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from andrewAdaptLibrary import ParkCatalog

NPS_SHP = os.path.join(
    PROJECT_ROOT,
    "USA_National_Park_Service_Lands_20170930_4993375350946852027",
    "USA_Federal_Lands.shp",
)
catalog = ParkCatalog(NPS_SHP)
print(catalog)

/opt/conda/envs/py-env/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: /workspaces/DSEBrandNew/USA_National_Park_Service_Lands_20170930_4993375350946852027/USA_Federal_Lands.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


ParkCatalog(437 NPS units)


## 1. search() gives find a park/area by name

Fuzzy matching, handles typos and partial names.

In [2]:
catalog.search("yosemite")

search: "yosemite"
   93.6  Yosemite National Park  *
   56.7  Allegheny Portage Railroad National Historic Site
   56.7  Amache National Historic Site
   56.7  Andersonville National Historic Site
   56.7  Andrew Johnson National Historic Site


[('Yosemite National Park', 93.63636363636364),
 ('Allegheny Portage Railroad National Historic Site', 56.666666666666664),
 ('Amache National Historic Site', 56.666666666666664),
 ('Andersonville National Historic Site', 56.666666666666664),
 ('Andrew Johnson National Historic Site', 56.666666666666664)]

In [3]:
catalog.search("rocky mountain")

search: "rocky mountain"
   95.0  Rocky Mountain National Park  *
   65.5  Great Smoky Mountains National Park
   61.4  Catoctin Mountain Park
   61.4  Kennesaw Mountain National Battlefield Park
   61.4  Kings Mountain National Military Park


[('Rocky Mountain National Park', 95.0),
 ('Great Smoky Mountains National Park', 65.5),
 ('Catoctin Mountain Park', 61.388888888888886),
 ('Kennesaw Mountain National Battlefield Park', 61.388888888888886),
 ('Kings Mountain National Military Park', 61.388888888888886)]

In [4]:
#even with typos
catalog.search("grand canion")

search: "grand canion"
   77.9  Grand Canyon National Park  *
   66.8  Grand Portage National Monument
   66.8  Grand Teton National Park
   62.9  Casa Grande Ruins National Monument
   59.9  Rio Grande Wild and Scenic River


[('Grand Canyon National Park', 77.91666666666667),
 ('Grand Portage National Monument', 66.78571428571428),
 ('Grand Teton National Park', 66.78571428571428),
 ('Casa Grande Ruins National Monument', 62.92207792207792),
 ('Rio Grande Wild and Scenic River', 59.88636363636364)]

## 2. what_is_available() gives back what data 

One needs to pass the exact name from search(). For parks inside the LOCA2 grid
(western US), this is fast, I've just hardcoded by info on CalAdapt (and working on this for all of last semester). For parks outside it,
this probes real Zarr stores on S3 with t2max at each timescale
(monthly, daily, hourly) to confirm data actually exists at that location since its more nebulous to find out whta exits and what doesnt'

In [5]:
# yosemite is in CA, inside LOCA2 grid -- fast
catalog.what_is_available("Yosemite National Park")

Yosemite National Park
  lat 37.49 to 38.19, lon -119.89 to -119.20
  inside LOCA2 grid -- all western US data available

  LOCA2 3km (statistical)
    scenarios:  historical, ssp245, ssp370, ssp585
    timescales: monthly, daily, yearly
    variables:  10
    models:     15
  WRF 3km (dynamical)
    scenarios:  historical, reanalysis, ssp370
    timescales: hourly, daily, monthly
    variables:  57
    models:     9
  WRF 9km (dynamical)
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    timescales: hourly, daily, monthly
    variables:  54
    models:     9
  WRF 45km (dynamical)
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    timescales: hourly, daily, monthly
    variables:  54
    models:     9



{'LOCA2_d03': {'name': 'LOCA2 3km (statistical)',
  'resolution': '3 km',
  'scenarios': ['historical', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['monthly', 'daily', 'yearly'],
  'n_variables': 10,
  'variables': ['tasmax',
   'tasmin',
   'pr',
   'hursmax',
   'hursmin',
   'huss',
   'rsds',
   'uas',
   'vas',
   'wspeed'],
  'n_models': 15},
 'WRF_d03': {'name': 'WRF 3km (dynamical)',
  'resolution': '3 km',
  'scenarios': ['historical', 'reanalysis', 'ssp370'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variables': 57,
  'n_models': 9},
 'WRF_d02': {'name': 'WRF 9km (dynamical)',
  'resolution': '9 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variables': 54,
  'n_models': 9},
 'WRF_d01': {'name': 'WRF 45km (dynamical)',
  'resolution': '45 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variabl

In [6]:
# rocky mountain is in colorado, outside LOCA2 -- probes WRF stores
catalog.what_is_available("Rocky Mountain National Park")

Rocky Mountain National Park
  lat 40.16 to 40.55, lon -105.91 to -105.49
  outside LOCA2 grid -- probed WRF stores with t2max

  WRF 45km (dynamical)
    monthly    1 grid cells, 1980-2014
    daily      1 grid cells, 1980-2014
    hourly     not in catalog
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    variables:  54
    models:     9
  WRF 9km (dynamical)
    monthly    17 grid cells, 1980-2014
    daily      17 grid cells, 1980-2014
    hourly     not in catalog
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    variables:  54
    models:     9



{'WRF_d01': {'name': 'WRF 45km (dynamical)',
  'resolution': '45 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'n_variables': 54,
  'n_models': 9,
  'probed': {'monthly': {'grid_cells': 1, 'years': '1980-2014'},
   'daily': {'grid_cells': 1, 'years': '1980-2014'},
   'hourly': None}},
 'WRF_d02': {'name': 'WRF 9km (dynamical)',
  'resolution': '9 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'n_variables': 54,
  'n_models': 9,
  'probed': {'monthly': {'grid_cells': 17, 'years': '1980-2014'},
   'daily': {'grid_cells': 17, 'years': '1980-2014'},
   'hourly': None}}}

In [7]:
# grand canyon is in AZ, inside LOCA2 grid
catalog.what_is_available("Grand Canyon National Park")

Grand Canyon National Park
  lat 35.75 to 36.87, lon -113.98 to -111.59
  inside LOCA2 grid -- all western US data available

  LOCA2 3km (statistical)
    scenarios:  historical, ssp245, ssp370, ssp585
    timescales: monthly, daily, yearly
    variables:  10
    models:     15
  WRF 3km (dynamical)
    scenarios:  historical, reanalysis, ssp370
    timescales: hourly, daily, monthly
    variables:  57
    models:     9
  WRF 9km (dynamical)
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    timescales: hourly, daily, monthly
    variables:  54
    models:     9
  WRF 45km (dynamical)
    scenarios:  historical, reanalysis, ssp245, ssp370, ssp585
    timescales: hourly, daily, monthly
    variables:  54
    models:     9



{'LOCA2_d03': {'name': 'LOCA2 3km (statistical)',
  'resolution': '3 km',
  'scenarios': ['historical', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['monthly', 'daily', 'yearly'],
  'n_variables': 10,
  'variables': ['tasmax',
   'tasmin',
   'pr',
   'hursmax',
   'hursmin',
   'huss',
   'rsds',
   'uas',
   'vas',
   'wspeed'],
  'n_models': 15},
 'WRF_d03': {'name': 'WRF 3km (dynamical)',
  'resolution': '3 km',
  'scenarios': ['historical', 'reanalysis', 'ssp370'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variables': 57,
  'n_models': 9},
 'WRF_d02': {'name': 'WRF 9km (dynamical)',
  'resolution': '9 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variables': 54,
  'n_models': 9},
 'WRF_d01': {'name': 'WRF 45km (dynamical)',
  'resolution': '45 km',
  'scenarios': ['historical', 'reanalysis', 'ssp245', 'ssp370', 'ssp585'],
  'timescales': ['hourly', 'daily', 'monthly'],
  'n_variabl